In [1]:
import logging
import time
import warnings
import shutil
import json
import pickle
import torch
import sys

import os
import pandas as pd
import numpy as np
import networkx as nx
import scanpy as sc
import anndata as ad
import torch.nn as nn

from datetime import datetime
from tqdm import tqdm
from collections import Counter
from itertools import islice

import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

# 1. 加载 autoreload 扩展
%load_ext autoreload

# 2. 设置模式为 "2" (表示自动重载所有模块)
%autoreload 2

import sys
sys.path.append("/home/liyang/BioWuYan/dygmamba_project/model/dygmamba")
sys.path.append("/home/liyang/BioWuYan/dygmamba_project/model/dygmamba/src")

from src.utils.load_configs import load_link_prediction_args
from src.data_preprocess import filter_jaspar_tf, adata_to_dataframe
from src.analysis.assess import dygmamba_assess

# Configuration

In [ ]:
import os
print("********************** start ********************")

start_time = time.time()  # start the time
print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Start the job")

args = load_link_prediction_args(is_evaluation=False)

print("**********************device********************")
print(f"Now use device is {args.device}")

org_data_path = "/home/liyang/BioWuYan/dygmamba_project/data/original/"

dyg_result_path = "/home/liyang/BioWuYan/dygmamba_project/data/dygmamba/res/result2/"

assess_result = "/home/liyang/BioWuYan/dygmamba_project/data/assess_result/"
os.makedirs(assess_result, exist_ok = True)

********************** start ********************
[2025-12-29 15:17:59] Start the job
**********************device********************
Now use device is cuda:0


# Load Data

In [5]:
adata_atac = ad.read_h5ad(dyg_result_path + "atac.h5ad")
adata_rna = ad.read_h5ad(dyg_result_path + "rna.h5ad")

# Result Analysis

## Benchmark

In [7]:

benchmark_tf_gene_threshold = 5
benchmark_data_path = org_data_path

benchmark_tf_gene_grn = ad.read_h5ad(benchmark_data_path + "tf_gene_network.h5ad")
benchmark_tf_gene_grn = benchmark_tf_gene_grn[:,adata_rna.var_names].copy()

benchmark_peak_gene_grn = ad.read_h5ad(benchmark_data_path + "peak_gene_network.h5ad")
benchmark_peak_gene_grn = benchmark_peak_gene_grn[adata_atac.var_names, adata_rna.var_names].copy()

benchmark_tf_peak_grn = ad.read_h5ad(benchmark_data_path + "tf_peak_network.h5ad")
benchmark_tf_peak_grn = benchmark_tf_peak_grn[adata_atac.var_names,:]

benchmark_tf_peak_df = adata_to_dataframe(benchmark_tf_peak_grn)
benchmark_tf_peak_df = benchmark_tf_peak_df.rename(columns= {"obs":"Peak", "var":"TF"})


benchmark_peak_gene_df = adata_to_dataframe(benchmark_peak_gene_grn)
benchmark_peak_gene_df = benchmark_peak_gene_df.rename(columns= {"obs":"Peak", "var":"Gene"})


benchmark_tf_gene_df = adata_to_dataframe(benchmark_tf_gene_grn)
benchmark_tf_gene_df = benchmark_tf_gene_df.rename(columns= {"obs":"TF", "var":"Gene"})


print(benchmark_peak_gene_grn)
print(benchmark_peak_gene_df.head())
print(f"Peak-Gene: {benchmark_peak_gene_df['Peak'].nunique()}, \
    {benchmark_peak_gene_df['Gene'].nunique()}, edge: {len(benchmark_peak_gene_df)}")

print(benchmark_tf_gene_grn)
print(benchmark_tf_gene_df.head())
print(f"TF-Gene: {benchmark_tf_gene_df['TF'].nunique()}, \
    {benchmark_tf_gene_df['Gene'].nunique()}, edge: {len(benchmark_tf_gene_df)}")

print(benchmark_tf_peak_grn)
print(benchmark_tf_peak_df.head())
print(f"Benchmark TF-Peak: {benchmark_tf_peak_df['TF'].nunique()}, \
    {benchmark_tf_peak_df['Peak'].nunique()}, edge: {len(benchmark_tf_peak_df)}")

AnnData object with n_obs × n_vars = 5000 × 500
    uns: 'description'
                     Peak   Gene  value
0  chr9-86353733-86354856   ACO2      1
1  chr9-86353733-86354856  CAMLG      1
2  chr9-86353733-86354856   LHX2      1
3  chr9-86353733-86354856  NINJ1      1
4  chr9-86353733-86354856   PGS1      1
Peak-Gene: 4972,     499, edge: 47382
AnnData object with n_obs × n_vars = 112 × 500
    uns: 'description'
       TF     Gene  value
0    RFX1  HNRNPA0     40
1     SP2  HNRNPA0    105
2     MGA  HNRNPA0    254
3   MEF2C  HNRNPA0    270
4  ZNF740  HNRNPA0    161
TF-Gene: 112,     500, edge: 55979
View of AnnData object with n_obs × n_vars = 5000 × 112
    uns: 'description'
                     Peak       TF  value
0  chr9-86353733-86354856     ARNT      1
1  chr9-86353733-86354856     ATF3      1
2  chr9-86353733-86354856     ATF7      1
3  chr9-86353733-86354856     BATF      1
4  chr9-86353733-86354856  BHLHE40      1
Benchmark TF-Peak: 112,     5000, edge: 437365


## TF-region data

In [6]:

jaspar_tf_region_file = org_data_path + "jaspar_data.h5ad"
jaspar_data = ad.read_h5ad(jaspar_tf_region_file)
adata_region_tf = filter_jaspar_tf(jaspar_data)


coo_matrix = adata_region_tf.X.tocoo()
tf_peak_df = pd.DataFrame({
    'Peak': adata_region_tf.obs_names[coo_matrix.row],
    'TF': adata_region_tf.var_names[coo_matrix.col],
    'value': coo_matrix.data
})
tf_peak_df = tf_peak_df[tf_peak_df["Peak"].isin(set(adata_atac.var_names))]

print("*"*50)
print(adata_region_tf)
print(tf_peak_df.head())
print("*"*50)
print(f"TF-Peak: {tf_peak_df['TF'].nunique()}, {tf_peak_df['Peak'].nunique()}, edge: {len(tf_peak_df)}")
print("*"*50)

/home/liyang/BioWuYan/conda_env/dygmamba39/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")



步骤 1: 过滤 Peaks (行)
  > 找到 72563 / 72584 个 peaks 至少有 1 个 TF 结合。


/home/liyang/BioWuYan/conda_env/dygmamba39/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")



步骤 2: 过滤 TFs (列)
  > 找到 879 / 879 个 TFs 至少结合 1 个 peak。


/home/liyang/BioWuYan/conda_env/dygmamba39/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


  > 最终形状: (72563, 879)
**************************************************
AnnData object with n_obs × n_vars = 72563 × 879
    uns: 'description'
                   Peak         TF  value
501  chr1-629315-630015  FOSB::JUN      1
502  chr1-629315-630015      Hmga1      1
503  chr1-629315-630015    BHLHA15      1
504  chr1-629315-630015      BATF3      1
505  chr1-629315-630015      Foxj2      1
**************************************************
TF-Peak: 828, 5000, edge: 3962265
**************************************************


In [9]:
dygmamba_tf_peak_df = tf_peak_df
dygmamba_tf_peak_df = dygmamba_tf_peak_df.rename(columns={"value":"predict"})

total_tf = set(benchmark_tf_peak_df["TF"]) & set(dygmamba_tf_peak_df["TF"])
total_peak = set(adata_atac.var_names)

benchmark_tf_peak_df = benchmark_tf_peak_df[benchmark_tf_peak_df["TF"].isin(total_tf)].copy()
dygmamba_tf_peak_df = dygmamba_tf_peak_df[dygmamba_tf_peak_df["TF"].isin(total_tf)].copy()

benchmark_tf_peak_df = benchmark_tf_peak_df[benchmark_tf_peak_df["Peak"].isin(total_peak)].copy()
dygmamba_tf_peak_df = dygmamba_tf_peak_df[dygmamba_tf_peak_df["Peak"].isin(total_peak)].copy()

dyg_merged_tf_peak_data = pd.merge(benchmark_tf_peak_df, dygmamba_tf_peak_df, on = ["TF", "Peak"], how="outer").fillna(0)

print(dyg_merged_tf_peak_data)
print(dygmamba_tf_peak_df)

print("*"*50)
print(f"Merged TF-Peak: {dyg_merged_tf_peak_data['TF'].nunique()}, {dyg_merged_tf_peak_data['Peak'].nunique()}, \
    {len(dyg_merged_tf_peak_data)}")

print(f"Dygmamba TF-Peak: {dygmamba_tf_peak_df['TF'].nunique()}, {dygmamba_tf_peak_df['Peak'].nunique()},\
    edge: {len(dygmamba_tf_peak_df)}")

print(f"Benchmark TF-Peak: {benchmark_tf_peak_df['TF'].nunique()}, \
    {benchmark_tf_peak_df['Peak'].nunique()}, edge: {len(benchmark_tf_peak_df)}")
print("*"*50)

                            Peak      TF  value  predict
0       chr1-100037313-100039097    ATF2    1.0      1.0
1         chr1-10032558-10033598    ATF2    0.0      1.0
2       chr1-100351277-100353494    ATF2    1.0      1.0
3       chr1-100894820-100896914    ATF2    0.0      1.0
4       chr1-101235525-101239028    ATF2    1.0      1.0
...                          ...     ...    ...      ...
533745    chrX-53683390-53684529  ZNF740    1.0      1.0
533746    chrX-53714219-53717438  ZNF740    0.0      1.0
533747    chrX-54043829-54044967  ZNF740    0.0      1.0
533748    chrX-64204939-64206473  ZNF740    1.0      1.0
533749      chrX-7147230-7148785  ZNF740    0.0      1.0

[533750 rows x 4 columns]
                              Peak      TF  predict
516             chr1-629315-630015   ESRRA        1
519             chr1-629315-630015     JUN        1
533             chr1-629315-630015   GABPA        1
535             chr1-629315-630015   NR2F1        1
539             chr1-629315-6

In [10]:

benchmark_result = []
result_type = "binary"
beta_value = 1

dyg_y_true = dyg_merged_tf_peak_data["value"].astype(int)
dyg_y_pre = dyg_merged_tf_peak_data["predict"].astype(int)
dyg_model_name = "Dygmamba_peak"

dyg_dict = dygmamba_assess(dyg_y_true, dyg_y_pre, model_name = dyg_model_name, 
                            beta = beta_value, type = result_type, fig_path = assess_result)
dyg_tf_region_result = pd.DataFrame([dyg_dict])
print("*"*50)
print(dyg_tf_region_result)



**************************************************
      model_name  TN     FP     FN      TP  precision    recall  FPR  \
0  Dygmamba_peak   0  96886  39486  397378   0.803979  0.909615  1.0   

        AUC   f_score  
0  0.454807  0.853541  


### Unibind score

In [24]:
import tarfile
import os
import io

def read_unibind_tf_peak(tar_file_path):

    # UniBind BED 文件的标准列名
    bed_columns = ['chrom', 'start', 'end', 'name', 'score', 'strand', 'signal', 'p_val', 'q_val', 'peak_center']

    # ================= 处理流程 =================
    all_tf_regions = []

    print(f"正在读取压缩包: {tar_file_path} ...")

    try:
        with tarfile.open(tar_file_path, "r") as tar:
            # 获取包内所有成员列表
            for member in tar.getmembers():

                # 1. 只处理 BED 文件
                if member.isfile() and member.name.endswith('.bed'):

                    # 2. 从文件名中提取 TF 名称
                    # UniBind 文件名通常格式: DATASET_ID.CELL_LINE.TF_NAME.MOTIF_ID.bed
                    # 例如: ENCSR000AKB.GM12878.CTCF.MA0139.1.bed
                    filename = os.path.basename(member.name)
                    parts = filename.split('.')

                    # 假设 TF 名称在第 3 个位置 (索引 2)，根据实际情况调整
                    # 这里的逻辑是寻找全大写的单词，或者依赖 UniBind 的命名规范
                    # 如果你有 metadata TSV 文件，最好配合那个用，这里演示纯文件名提取
                    if len(parts) >= 3:
                        tf_name = parts[2]
                    else:
                        tf_name = "Unknown"

                    # 3. 读取 BED 文件内容
                    f = tar.extractfile(member)
                    if f:
                        # 使用 io.TextIOWrapper 将字节流转为文本流
                        content = io.TextIOWrapper(f, encoding='utf-8')

                        # 读取数据
                        df = pd.read_csv(content, sep='\t', header=None, names=bed_columns)

                        # 只保留核心坐标列，节省内存
                        df = df[['chrom', 'start', 'end']]

                        # 添加 TF 标签
                        df['TF'] = tf_name

                        # 添加源文件 ID (可选，用于追溯)
                        df['SourceID'] = parts[0]

                        all_tf_regions.append(df)

        # 4. 合并所有数据
        if all_tf_regions:
            unibind_df = pd.concat(all_tf_regions, ignore_index=True)
            print("处理完成！")
            print(f"共提取了 {unibind_df['TF'].nunique()} 个 TF 的数据。")
            print(f"总 Region 数量: {len(unibind_df)}")
            print(unibind_df.head())

            # 可选：保存为 CSV 备用
            # unibind_df.to_csv("GM12878_UniBind_Regions.csv", index=False)
        else:
            print("警告：未在压缩包中找到 .bed 文件。")

    except FileNotFoundError:
        print("错误：找不到指定的 tar 文件，请检查路径。")
        
    return unibind_df

import pandas as pd
import pybedtools
import os

# 避免 pybedtools 产生过多临时文件警告
pybedtools.helpers.set_tempdir('/tmp') 

def calculate_tf_metrics(pred_df, unibind_df):
    """
    计算 TF-Region 预测的 Precision, Recall, F1
    
    参数:
    pred_df: 包含 SCENIC+ 预测结果的 DataFrame
             必须列: 'TF', 'peak' (格式如 chr1-100-200)
             
    unibind_df: 包含 UniBind 金标准的 DataFrame
                必须列: 'TF', 'chrom', 'start', 'end'
    """
    
    # ---------------------------------------------------------
    # 1. 预处理预测数据 (解析 Peak 坐标)
    # ---------------------------------------------------------
    print("正在解析 Consensus Peaks 坐标...")
    
    # 提取所有唯一的 Consensus Peaks (作为全集/背景)
    all_consensus_peaks = pred_df['Peak'].unique()
    consensus_map_df = pd.DataFrame({'peak_id': all_consensus_peaks})
    
    # 解析 "chr1-start-end" 格式
    coords = consensus_map_df['peak_id'].str.extract(r'(?P<chrom>.+)-(?P<start>\d+)-(?P<end>\d+)')
    consensus_map_df = pd.concat([consensus_map_df, coords], axis=1)
    
    # 转换坐标类型
    consensus_map_df['start'] = consensus_map_df['start'].astype(int)
    consensus_map_df['end'] = consensus_map_df['end'].astype(int)
    
    consensus_bed_df = consensus_map_df[['chrom', 'start', 'end', 'peak_id']]
    consensus_bed_all = pybedtools.BedTool.from_dataframe(consensus_bed_df)
    
    # ---------------------------------------------------------
    # 2. 逐个 TF 计算指标
    # ---------------------------------------------------------
    # 找出两个数据集中共有的 TF
    common_tfs = set(pred_df['TF'].unique()) & set(unibind_df['TF'].unique())
    print(f"共发现 {len(common_tfs)} 个共有 TF，开始评估...")
    
    metrics_list = []
    
    for tf in common_tfs:
        # --- A. 准备真值 (Ground Truth) ---
        # 获取该 TF 在 UniBind 中的所有物理结合区域
        tf_unibind_subset = unibind_df[unibind_df['TF'] == tf][['chrom', 'start', 'end']]
        
        if tf_unibind_subset.empty:
            continue
            
        tf_unibind_bed = pybedtools.BedTool.from_dataframe(tf_unibind_subset)
        
        # --- B. 确定真值集合 (Ground Truth Set of Consensus Peaks) ---
        # 核心逻辑：如果一个 Consensus Peak 与 UniBind 区域有重叠，它就是该 TF 的"真"靶点
        # -u: 只要有重叠就输出
        # -wa: 输出原始的 A (即 Consensus Peak)
        true_targets_bed = consensus_bed_all.intersect(tf_unibind_bed, u=True, wa=True)
        
        # 获取真值 Peak ID 集合
        true_peak_ids = set([f.name for f in true_targets_bed])
        
        # --- C. 获取预测集合 (Predicted Set) ---
        # SCENIC+ 预测该 TF 结合的 Consensus Peaks
        pred_peak_ids = set(pred_df[pred_df['TF'] == tf]['Peak'])
        
        # --- D. 计算指标 ---
        # TP: 预测了，且是真的
        tp = len(pred_peak_ids.intersection(true_peak_ids))
        
        # FP: 预测了，但不是真的
        fp = len(pred_peak_ids) - tp
        
        # FN: 没预测，但是真的 (在 true_peak_ids 里，但不在 pred_peak_ids 里)
        fn = len(true_peak_ids) - tp
        
        # 防止除以 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        metrics_list.append({
            'TF': tf,
            'Precision': precision,
            'Recall': recall,
            'F1': f1,
            'TP': tp,
            'FP': fp,
            'FN': fn,
            'GroundTruth_Count': len(true_peak_ids),
            'Predicted_Count': len(pred_peak_ids)
        })
        
    # ---------------------------------------------------------
    # 3. 汇总结果
    # ---------------------------------------------------------
    metrics_df = pd.DataFrame(metrics_list)
    
    # 清理临时文件 (pybedtools 习惯)
    pybedtools.cleanup()
    
    return metrics_df

In [ ]:
import pandas as pd

all_consensus_peaks = tf_peak_df['Peak'].unique()
consensus_df = pd.DataFrame({'peak_id': all_consensus_peaks})

coords = consensus_df['peak_id'].str.extract(r'(?P<chrom>.+)-(?P<start>\d+)-(?P<end>\d+)')
consensus_df = pd.concat([consensus_df, coords], axis=1)
consensus_df['start'] = consensus_df['start'].astype(int)
consensus_df['end'] = consensus_df['end'].astype(int)
print(consensus_df)

tf_chip_seq_scenic = pd.read_parquet("/home/liyang/BioWuYan/MethodTest/Data/All/SCENIC_plus/combined_tf_peaks_robust.parquet")
cell_type = "GM12878"
cell_line_ChIP_seq = tf_chip_seq_scenic[tf_chip_seq_scenic["cell_type"]==cell_type]
cell_line_tf_set = set(cell_line_ChIP_seq["tf_name"])

print(f"The cell line TF have: {len(cell_line_tf_set)}, they are: {cell_line_tf_set}")

unibind_file = "/home/liyang/BioWuYan/dygmamba_project/data/Unibind/GM12878_UniBind_search_0_998agt.tar.gz"
unibind_tf_peak_df = read_unibind_tf_peak(unibind_file)
print(f"before filter: {len(unibind_tf_peak_df)}")
unibind_tf_peak_df = unibind_tf_peak_df[unibind_tf_peak_df["TF"].isin(cell_line_tf_set)].copy()
unibind_tf_peak_df


                       peak_id chrom      start        end
0           chr1-629315-630015  chr1     629315     630015
1           chr1-630813-631351  chr1     630813     631351
2           chr1-633902-634203  chr1     633902     634203
3           chr1-826900-828028  chr1     826900     828028
4         chr1-1019255-1020913  chr1    1019255    1020913
...                        ...   ...        ...        ...
4995  chrX-154341456-154343917  chrX  154341456  154343917
4996  chrX-154378369-154379460  chrX  154378369  154379460
4997  chrX-154388184-154390842  chrX  154388184  154390842
4998  chrX-154546742-154547974  chrX  154546742  154547974
4999  chrX-155026675-155027933  chrX  155026675  155027933

[5000 rows x 4 columns]


In [26]:
# results = calculate_tf_metrics(tf_peak_df, unibind_tf_peak_df)
print(results)


         TF  Precision    Recall        F1    TP    FP    FN  \
0      PAX5   0.919184  1.000000  0.957890  4595   404     0   
1      JUND   0.161649  0.992405  0.278014   784  4066     6   
2     NR2C2   0.041408  1.000000  0.079524   207  4792     0   
3     CREB1   0.706483  0.999717  0.827902  3531  1467     1   
4      CUX1   0.005101  1.000000  0.010150    24  4681     0   
5     CEBPB   0.226752  0.716298  0.344461   356  1214   141   
6      ELF1   0.855800  1.000000  0.922298  4279   721     0   
7      BATF   0.542709  1.000000  0.703579  2713  2286     0   
8      NFYA   0.100200  1.000000  0.182149   501  4499     0   
9      MAFK   0.002400  1.000000  0.004789    12  4987     0   
10  BHLHE40   0.814248  0.999753  0.897516  4046   923     1   
11     EGR1   0.762953  1.000000  0.865540  3814  1185     0   
12    TBX21   0.393479  1.000000  0.564743  1967  3032     0   
13     JUNB   0.153342  1.000000  0.265908   748  4130     0   
14     IRF3   0.194476  0.441253  0.2699

In [28]:
print(cell_line_tf_set)

{'SPI1', 'ARNT', 'PAX5', 'JUND', 'CREB1', 'NR2C2', 'CUX1', 'ELF1', 'CEBPB', 'BATF', 'NFYA', 'MAFK', 'BHLHE40', 'EGR1', 'ATF2', 'JUNB', 'TBX21', 'IRF3', 'MEF2A', 'NR2F1', 'HSF1', 'EBF1', 'CTCF', 'MYC', 'ETV6', 'TCF7', 'PBX3', 'RUNX3', 'ESRRA', 'IRF4', 'ATF3', 'E2F4', 'USF2', 'MXI1', 'SRF', 'USF1', 'YY1', 'REST', 'RXRA', 'NFYB', 'TCF12', 'ATF7', 'RELA', 'CREM', 'TCF3', 'MEF2C', 'GABPA', 'ETS1', 'NR2C1'}


## Region-gene

### hicstraw

In [11]:

adata_rp_gene_peak = ad.read_h5ad(dyg_result_path + "rp_gene_peak.h5ad")
prior_peak_gene_df = adata_to_dataframe(adata_rp_gene_peak)
prior_peak_gene_df = prior_peak_gene_df.rename(columns= {"obs":"Gene", "var":"Peak"})

print("*"*50)
print(adata_rp_gene_peak)
print(prior_peak_gene_df.head())
print(f"Prior peak-gene: {prior_peak_gene_df['Peak'].nunique()}, {prior_peak_gene_df['Gene'].nunique()},\
    edge: {len(prior_peak_gene_df)}")

**************************************************
AnnData object with n_obs × n_vars = 500 × 5000
    uns: 'decay_distance', 'description', 'max_range'
      Gene                       Peak  value
0   NDUFS5     chr1-38990697-38992620      1
1     DPP9      chr19-4790997-4792145      1
2  TXNDC15   chr5-134904464-134905833      1
3    PPRC1  chr10-102055526-102056382      1
4    PPRC1  chr10-102064962-102066074      1
Prior peak-gene: 412, 246,    edge: 442


In [12]:
Node_id = pd.read_pickle(dyg_result_path + "node_id.pkl")
graph_df = pd.read_pickle(dyg_result_path + "Graph_df.pkl")
graph_df["Unnamed"] = graph_df.index
name_list = ["Unnamed", "source_node", "target_node", "time", "label", "edge_idx"]
New_Graph = graph_df[name_list].copy()
New_Graph.columns = ['Unnamed: 0', 'u', 'i', 'ts', 'label', 'idx']

result_path = dyg_result_path + 'my_result_run{run}.npy'
predict_edge_label = np.load(result_path)

# binary_output = (predict_edge_label > 0.5).astype(int)
New_Graph["predict"] = predict_edge_label
predict_grn = New_Graph.copy()

mapping_series = Node_id["name"]
predict_grn['source'] = (predict_grn['u'] - 1).map(mapping_series)
predict_grn['target'] = (predict_grn['i'] - 1).map(mapping_series)

peak_gene_df = predict_grn[['source', 'target', 'ts','predict']].rename(
    columns={'source': 'Peak', 'target': 'Gene'}
)
peak_gene_df = peak_gene_df[~peak_gene_df["Gene"].str.startswith('chr')].copy()

print("*"*50)
print(peak_gene_df.head())
print(f"Peak-Gene: {peak_gene_df['Peak'].nunique()}, {peak_gene_df['Gene'].nunique()}, edge:{len(peak_gene_df)}")
print("*"*50)


**************************************************
                      Peak  Gene        ts   predict
0  chr12-53370831-53371774  AAAS  0.103799  0.054033
1  chr12-53370831-53371774  AAAS  0.130393  1.000000
2  chr12-53370831-53371774  AAAS  0.427715  1.000000
3  chr12-53370831-53371774  AAAS  0.487691  1.000000
4  chr12-53370831-53371774  AAAS  0.560851  1.000000
Peak-Gene: 412, 246, edge:37228
**************************************************


In [13]:
dygmamba_peak_gene_grn = peak_gene_df

avg_active_peak_gene_grn = dygmamba_peak_gene_grn.groupby(['Peak', 'Gene']).agg(
    avg_ts_weight=('predict', 'mean'),  # 对 weight 列做均值，新列名叫 avg_weight
    avg_total_weight=('predict', 'sum')  # (可选) 建议顺便算个总权重
).reset_index()

avg_active_peak_gene_grn = avg_active_peak_gene_grn[avg_active_peak_gene_grn["Peak"].isin(total_peak)].copy()
benchmark_peak_gene_df = benchmark_peak_gene_df[benchmark_peak_gene_df["Peak"].isin(total_peak)].copy()

dyg_merged_peak_gene_data = pd.merge(benchmark_peak_gene_df, avg_active_peak_gene_grn, on = ["Gene", "Peak"], how="outer").fillna(0)

print("*"*50)
print(f"Merged Peak-Gene: {dyg_merged_peak_gene_data['Peak'].nunique()}, {dyg_merged_peak_gene_data['Gene'].nunique()}, \
    {len(dyg_merged_peak_gene_data)}")

print(f"Dygmamba Peak-Gene: {avg_active_peak_gene_grn['Peak'].nunique()}, {avg_active_peak_gene_grn['Gene'].nunique()}, \
    edge: {len(avg_active_peak_gene_grn)}")

print(f"Benchmark Peak-Gene: {benchmark_peak_gene_df['Peak'].nunique()}, \
    {benchmark_peak_gene_df['Gene'].nunique()}, edge: {len(benchmark_peak_gene_df)}")
print("*"*50)



**************************************************
Merged Peak-Gene: 4972, 499,     47382
Dygmamba Peak-Gene: 412, 246,     edge: 442
Benchmark Peak-Gene: 4972,     499, edge: 47382
**************************************************


In [14]:

dyg_merged_peak_gene_data["predict"] = (dyg_merged_peak_gene_data["avg_ts_weight"]>0.9).astype(int)

dyg_y_true = dyg_merged_peak_gene_data["value"].astype(int)
dyg_y_pre = dyg_merged_peak_gene_data["predict"].astype(int)
dyg_model_name = "Dygmamba_peak_gene"

dyg_dict = dygmamba_assess(dyg_y_true, dyg_y_pre, model_name = dyg_model_name, 
                            beta = beta_value, type = result_type, fig_path = assess_result)
dyg_peak_gene_result = pd.DataFrame([dyg_dict])

print("*"*50)
print(dyg_peak_gene_result)
print(f"merged: {len(dyg_merged_peak_gene_data)}, benchmark peak gene: {len(benchmark_peak_gene_df)},\
    dyg peak gene {len(avg_active_peak_gene_grn)}")
print(f"Peak-Gene: {dyg_merged_peak_gene_data['Peak'].nunique()}, \
    {dyg_merged_peak_gene_data['Gene'].nunique()}, edge:{len(dyg_merged_peak_gene_data)}")
print("*"*50)

**************************************************
           model_name  TN  FP     FN   TP  precision    recall  FPR  AUC  \
0  Dygmamba_peak_gene   0   0  46940  442        1.0  0.009328  NaN  NaN   

    f_score  
0  0.018484  
merged: 47382, benchmark peak gene: 47382,    dyg peak gene 442
Peak-Gene: 4972,     499, edge:47382
**************************************************


/home/liyang/BioWuYan/dygmamba_project/model/dygmamba/src/analysis/assess.py:56: RuntimeWarning: invalid value encountered in scalar divide
  model_FPR = float(model_FP/(model_FP+model_TN))


### scenic+

## TF-gene network

In [15]:
dygmamba_tf_peak_df = dygmamba_tf_peak_df.rename(columns={"predict":"value"})
peak_gene_grn = peak_gene_df[peak_gene_df["Peak"].isin(total_peak)]
merged_df = pd.merge(dygmamba_tf_peak_df, peak_gene_grn, on='Peak')

tf_gene_grn = merged_df.groupby(['TF', 'Gene', 'ts']).agg(
    peak_num=('Peak', 'nunique'),   # 对 Peak 列做去重计数，新列名叫 peak_num
    avg_weight=('predict', 'mean'),  # 对 weight 列做均值，新列名叫 avg_weight
    total_weight=('predict', 'sum')  # (可选) 建议顺便算个总权重
).reset_index()

# 查看结果
print("*"*50)
print(tf_gene_grn.head())

print(f"Dygmamba TF-Peak: {dygmamba_tf_peak_df['TF'].nunique()}, {dygmamba_tf_peak_df['Peak'].nunique()},\
    edge: {len(dygmamba_tf_peak_df)}")

print(f"Peak-Gene: {peak_gene_df['Peak'].nunique()}, {peak_gene_df['Gene'].nunique()}, edge:{len(peak_gene_df)}")

print(f"TF-Gene: {tf_gene_grn['TF'].nunique()}, {tf_gene_grn['Gene'].nunique()}, edge: {len(tf_gene_grn)}")

tf_gene_grn.to_pickle(dyg_result_path + "new_tf_gene_grn_1224.pkl")


**************************************************
     TF  Gene        ts  peak_num  avg_weight  total_weight
0  ATF2  AAAS  0.000000         1    0.064111      0.064111
1  ATF2  AAAS  0.103799         2    0.527016      1.054032
2  ATF2  AAAS  0.130393         2    1.000000      2.000000
3  ATF2  AAAS  0.427715         2    1.000000      2.000000
4  ATF2  AAAS  0.487691         2    1.000000      2.000000
Dygmamba TF-Peak: 98, 5000,    edge: 494264
Peak-Gene: 412, 246, edge:37228
TF-Gene: 98, 246, edge: 2510271


### Average GRN


In [16]:
avg_active_tf_gene_grn = tf_gene_grn.groupby(['TF', 'Gene']).agg(
    avg_ts_weight=('total_weight', 'mean'),  # 对 weight 列做均值，新列名叫 avg_weight
    avg_total_weight=('total_weight', 'sum')  # (可选) 建议顺便算个总权重
).reset_index()
avg_active_tf_gene_grn.to_pickle(dyg_result_path + "average_active_tf_gene_grn.pkl")




pivoted_grn = tf_gene_grn.pivot_table(
    index=['TF', 'Gene'],
    columns='ts',
    values='total_weight',
    fill_value=0
)
pivoted_grn['average_active_weight'] = pivoted_grn.mean(axis=1)
avg_global_tf_gene_grn = pivoted_grn.reset_index()
avg_global_tf_gene_grn = avg_global_tf_gene_grn[["TF","Gene","average_active_weight"]].copy()
avg_global_tf_gene_grn.columns.name = None
avg_global_tf_gene_grn.to_pickle(dyg_result_path + "average_global_tf_gene_grn.pkl")

print("*"*50)
print(avg_active_tf_gene_grn.head())
print(avg_global_tf_gene_grn.head())

print("*"*50)
print(f"TF-Gene: {tf_gene_grn['TF'].nunique()}, {tf_gene_grn['Gene'].nunique()}, edge: {len(tf_gene_grn)}")
print(f"avg active TF-Gene: {avg_active_tf_gene_grn['TF'].nunique()}, {avg_active_tf_gene_grn['Gene'].nunique()},\
    edge: {len(avg_active_tf_gene_grn)}")
print(f"avg global TF-Gene: {avg_global_tf_gene_grn['TF'].nunique()}, {avg_global_tf_gene_grn['Gene'].nunique()},\
    edge: {len(avg_global_tf_gene_grn)}")
print("*"*50)

**************************************************
     TF      Gene  avg_ts_weight  avg_total_weight
0  ATF2      AAAS       1.333969        248.118149
1  ATF2  AASDHPPT       0.987846         76.064110
2  ATF2     ABHD6       0.983581         56.064110
3  ATF2      ACO2       1.557796        288.192322
4  ATF2     ADCY3       0.989601         89.064110
     TF      Gene  average_active_weight
0  ATF2      AAAS               1.112637
1  ATF2  AASDHPPT               0.341095
2  ATF2     ABHD6               0.251409
3  ATF2      ACO2               1.292342
4  ATF2     ADCY3               0.399391
**************************************************
TF-Gene: 98, 246, edge: 2510271
avg active TF-Gene: 98, 246,    edge: 21958
avg global TF-Gene: 98, 246,    edge: 21958
**************************************************


### Merge

#### benchmark merge

In [39]:
benchmark_tf_gene_df_new = pd.merge(benchmark_tf_peak_df, benchmark_peak_gene_df, on='Peak')
benchmark_tf_gene_grn_new = benchmark_tf_gene_df_new.groupby(['TF', 'Gene']).agg(
    peak_num=('Peak', 'nunique'),   # 对 Peak 列做去重计数，新列名叫 peak_num
).reset_index()
benchmark_tf_gene_grn_new = benchmark_tf_gene_grn_new.rename(columns= {"peak_num":"value"})

print("*"*50)
print(benchmark_tf_gene_df_new.head())
print(benchmark_tf_gene_grn_new.head())
print("*"*50)
print(f"Benchmark TF-Gene: {benchmark_tf_gene_df['TF'].nunique()}, \
    {benchmark_tf_gene_df['Gene'].nunique()}, edge: {len(benchmark_tf_gene_df)}")
print(f"New Benchmark TF-Gene: {benchmark_tf_gene_df_new['TF'].nunique()}, \
    {benchmark_tf_gene_df_new['Gene'].nunique()}, edge: {len(benchmark_tf_gene_df_new)}")
print(f"New count Benchmark TF-Gene: {benchmark_tf_gene_grn_new['TF'].nunique()}, \
    {benchmark_tf_gene_grn_new['Gene'].nunique()}, edge: {len(benchmark_tf_gene_grn_new)}")
print("*"*50)


benchmark_tf_gene_grn_new = benchmark_tf_gene_grn_new[benchmark_tf_gene_grn_new["TF"].isin(total_tf)].copy()

dyg_avg_active_tf_gene_grn = avg_active_tf_gene_grn
dyg_avg_global_tf_gene_grn = avg_global_tf_gene_grn

dyg_avg_active_tf_gene_grn.columns.name = ""
dyg_avg_active_tf_gene_grn = dyg_avg_active_tf_gene_grn.reset_index()
dyg_avg_active_tf_gene_grn = dyg_avg_active_tf_gene_grn.drop(["index"],axis = 1)
dyg_merged_data = pd.merge(benchmark_tf_gene_grn_new, dyg_avg_active_tf_gene_grn, on = ["TF", "Gene"], how="outer").fillna(0)

dyg_avg_global_tf_gene_grn.columns.name = ""
dyg_avg_global_tf_gene_grn = dyg_avg_global_tf_gene_grn.reset_index()
dyg_avg_global_tf_gene_grn = dyg_avg_global_tf_gene_grn.drop(["index"],axis = 1)
dyg_global_merged_data = pd.merge(benchmark_tf_gene_grn_new, dyg_avg_global_tf_gene_grn, on = ["TF", "Gene"], how="outer").fillna(0)


benchmark_tf_gene_threshold = 20
threshold_weight_global = 0.2
threshold_weight_active = 0.2
benchmark_result = []
result_type = "binary"
beta_value = 1

dyg_merged_data["label"] = (dyg_merged_data["value"] > benchmark_tf_gene_threshold).astype(int)
dyg_merged_data["predict_label"] = (dyg_merged_data["avg_ts_weight"]> threshold_weight_active).astype(int)
dyg_merge_grn = dyg_merged_data.copy()
dyg_y_true = dyg_merge_grn["label"].astype(int)
dyg_y_pre = dyg_merge_grn["predict_label"].astype(int)
dyg_model_name = "Dygmamba"
dyg_dict = dygmamba_assess(dyg_y_true, dyg_y_pre, model_name = dyg_model_name, 
                            beta = beta_value, type = result_type, fig_path = assess_result)

benchmark_result.append(dyg_dict)

dyg_global_merged_data["label"] = (dyg_global_merged_data["value"] > benchmark_tf_gene_threshold).astype(int)
dyg_global_merged_data["predict_label"] = (dyg_global_merged_data["average_active_weight"]> threshold_weight_global).astype(int)
dyg_global_merge_grn = dyg_global_merged_data.copy()
dyg_global_y_true = dyg_global_merge_grn["label"].astype(int)
dyg_global_y_pre = dyg_global_merge_grn["predict_label"].astype(int)
dyg_global_model_name = "Dygmamba" + "_global"
dyg_global_dict = dygmamba_assess(dyg_global_y_true, dyg_global_y_pre, model_name = dyg_global_model_name, 
                            beta = beta_value, type = result_type, fig_path = assess_result)

benchmark_result.append(dyg_global_dict)
    
benchmark_result_df = pd.DataFrame(benchmark_result)

print(benchmark_result_df)

active_num = dyg_merged_data["label"].sum(axis=0)
active_total = len(dyg_merged_data)

global_num = dyg_global_merged_data["label"].sum(axis=0)
global_total = len(dyg_global_merged_data)

print(f"active_num: {active_num}/{active_total}; global num: {global_num}/{global_total}")

print("*"*50)

print(f"Benchmark TF-Gene: {benchmark_tf_gene_grn_new['TF'].nunique()}, \
    {benchmark_tf_gene_grn_new['Gene'].nunique()}, edge: {len(benchmark_tf_gene_grn_new)}")

print(f"Dygmamba Merge TF-Gene: {dyg_merge_grn['TF'].nunique()}, \
    {dyg_merge_grn['Gene'].nunique()}, edge: {len(dyg_merge_grn)}")

print(f"Dygmamba Global Merge TF-Gene: {dyg_global_merged_data['TF'].nunique()}, \
    {dyg_global_merged_data['Gene'].nunique()}, edge: {len(dyg_global_merged_data)}")

print(f"Dygmamba Global TF-Gene: {dyg_avg_global_tf_gene_grn['TF'].nunique()}, \
    {dyg_avg_global_tf_gene_grn['Gene'].nunique()}, edge: {len(dyg_avg_global_tf_gene_grn)}")

print(f"Dygmamba active TF-Gene: {dyg_avg_active_tf_gene_grn['TF'].nunique()}, \
    {dyg_avg_active_tf_gene_grn['Gene'].nunique()}, edge: {len(dyg_avg_active_tf_gene_grn)}")
print("*"*50)



/tmp/ipykernel_9210/3338224131.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  benchmark_tf_gene_grn_new = benchmark_tf_gene_df_new.groupby(['TF', 'Gene']).agg(


**************************************************
                     Peak    TF  value_x   Gene  value_y
0  chr9-86353733-86354856  ATF3        1   ACO2        1
1  chr9-86353733-86354856  ATF3        1  CAMLG        1
2  chr9-86353733-86354856  ATF3        1   LHX2        1
3  chr9-86353733-86354856  ATF3        1  NINJ1        1
4  chr9-86353733-86354856  ATF3        1   PGS1        1
     TF      Gene  value
0  ARNT      AAAS      0
1  ARNT      AAR2      0
2  ARNT  AASDHPPT      0
3  ARNT     ABHD6      0
4  ARNT     ACADM      0
**************************************************
Benchmark TF-Gene: 112,     500, edge: 55979
New Benchmark TF-Gene: 98,     499, edge: 3667137
New count Benchmark TF-Gene: 112,     499, edge: 55888
**************************************************
        model_name    TN   FP     FN     TP  precision    recall       FPR  \
0         Dygmamba  2588  365  24356  21593   0.983377  0.469934  0.123603   
1  Dygmamba_global  2588  365  24356  21593   0.9

In [38]:
merge_data2 = pd.merge(benchmark_tf_gene_grn_new, benchmark_tf_gene_df, on=['TF', 'Gene'])
print(benchmark_tf_gene_grn_new)
print(benchmark_tf_gene_df)
print(merge_data2)

           TF      Gene  value
499      ATF2      AAAS     86
500      ATF2      AAR2     39
501      ATF2  AASDHPPT     29
502      ATF2     ABHD6     45
503      ATF2     ACADM     34
...       ...       ...    ...
55883  ZNF740    ZNF672     40
55884  ZNF740    ZNF697     24
55885  ZNF740    ZNF721     47
55886  ZNF740     ZNF76     47
55887  ZNF740    ZSWIM1     68

[48902 rows x 3 columns]
           TF     Gene  value
0        RFX1  HNRNPA0     40
1         SP2  HNRNPA0    105
2         MGA  HNRNPA0    254
3       MEF2C  HNRNPA0    270
4      ZNF740  HNRNPA0    161
...       ...      ...    ...
55974    BATF  METTL14    285
55975   BACH1  METTL14    133
55976    ATF4  METTL14    107
55977    ATF3  METTL14    190
55978    ARNT  METTL14    260

[55979 rows x 3 columns]
           TF      Gene  value_x  value_y
0        ATF2      AAAS       86      283
1        ATF2      AAR2       39      201
2        ATF2  AASDHPPT       29      172
3        ATF2     ABHD6       45      214
4     

#### h5ad

In [30]:
benchmark_tf_gene_grn_new = benchmark_tf_gene_df.copy()
print(f"New count Benchmark TF-Gene: {benchmark_tf_gene_grn_new['TF'].nunique()}, \
    {benchmark_tf_gene_grn_new['Gene'].nunique()}, edge: {len(benchmark_tf_gene_grn_new)}")
benchmark_tf_gene_grn_new = benchmark_tf_gene_grn_new[benchmark_tf_gene_grn_new["TF"].isin(total_tf)].copy()

dyg_avg_active_tf_gene_grn = avg_active_tf_gene_grn
dyg_avg_global_tf_gene_grn = avg_global_tf_gene_grn

dyg_avg_active_tf_gene_grn.columns.name = ""
dyg_avg_active_tf_gene_grn = dyg_avg_active_tf_gene_grn.reset_index()
dyg_avg_active_tf_gene_grn = dyg_avg_active_tf_gene_grn.drop(["index"],axis = 1)
dyg_merged_data = pd.merge(benchmark_tf_gene_grn_new, dyg_avg_active_tf_gene_grn, on = ["TF", "Gene"], how="outer").fillna(0)

dyg_avg_global_tf_gene_grn.columns.name = ""
dyg_avg_global_tf_gene_grn = dyg_avg_global_tf_gene_grn.reset_index()
dyg_avg_global_tf_gene_grn = dyg_avg_global_tf_gene_grn.drop(["index"],axis = 1)
dyg_global_merged_data = pd.merge(benchmark_tf_gene_grn_new, dyg_avg_global_tf_gene_grn, on = ["TF", "Gene"], how="outer").fillna(0)


benchmark_tf_gene_threshold = 200
threshold_weight_global = 0.2
threshold_weight_active = 0.2
benchmark_result = []
result_type = "binary"
beta_value = 1

dyg_merged_data["label"] = (dyg_merged_data["value"] > benchmark_tf_gene_threshold).astype(int)
dyg_merged_data["predict_label"] = (dyg_merged_data["avg_ts_weight"]> threshold_weight_active).astype(int)
dyg_merge_grn = dyg_merged_data.copy()
dyg_y_true = dyg_merge_grn["label"].astype(int)
dyg_y_pre = dyg_merge_grn["predict_label"].astype(int)
dyg_model_name = "Dygmamba"
dyg_dict = dygmamba_assess(dyg_y_true, dyg_y_pre, model_name = dyg_model_name, 
                            beta = beta_value, type = result_type, fig_path = assess_result)

benchmark_result.append(dyg_dict)

dyg_global_merged_data["label"] = (dyg_global_merged_data["value"] > benchmark_tf_gene_threshold).astype(int)
dyg_global_merged_data["predict_label"] = (dyg_global_merged_data["average_active_weight"]> threshold_weight_global).astype(int)
dyg_global_merge_grn = dyg_global_merged_data.copy()
dyg_global_y_true = dyg_global_merge_grn["label"].astype(int)
dyg_global_y_pre = dyg_global_merge_grn["predict_label"].astype(int)
dyg_global_model_name = "Dygmamba" + "_global"
dyg_global_dict = dygmamba_assess(dyg_global_y_true, dyg_global_y_pre, model_name = dyg_global_model_name, 
                            beta = beta_value, type = result_type, fig_path = assess_result)

benchmark_result.append(dyg_global_dict)
    
benchmark_result_df = pd.DataFrame(benchmark_result)

print(benchmark_result_df)


active_num = dyg_merged_data["label"].sum(axis=0)
active_total = len(dyg_merged_data)

global_num = dyg_global_merged_data["label"].sum(axis=0)
global_total = len(dyg_global_merged_data)

print(f"active_num: {active_num}/{active_total}; global num: {global_num}/{global_total}")

print("*"*50)

print(f"Benchmark TF-Gene: {benchmark_tf_gene_grn_new['TF'].nunique()}, \
    {benchmark_tf_gene_grn_new['Gene'].nunique()}, edge: {len(benchmark_tf_gene_grn_new)}")

print(f"Dygmamba Merge TF-Gene: {dyg_merge_grn['TF'].nunique()}, \
    {dyg_merge_grn['Gene'].nunique()}, edge: {len(dyg_merge_grn)}")

print(f"Dygmamba Global Merge TF-Gene: {dyg_global_merged_data['TF'].nunique()}, \
    {dyg_global_merged_data['Gene'].nunique()}, edge: {len(dyg_global_merged_data)}")

print(f"Dygmamba Global TF-Gene: {dyg_avg_global_tf_gene_grn['TF'].nunique()}, \
    {dyg_avg_global_tf_gene_grn['Gene'].nunique()}, edge: {len(dyg_avg_global_tf_gene_grn)}")

print(f"Dygmamba active TF-Gene: {dyg_avg_active_tf_gene_grn['TF'].nunique()}, \
    {dyg_avg_active_tf_gene_grn['Gene'].nunique()}, edge: {len(dyg_avg_active_tf_gene_grn)}")
print("*"*50)



New count Benchmark TF-Gene: 112,     500, edge: 55979
        model_name     TN    FP     FN     TP  precision    recall       FPR  \
0         Dygmamba  10382  3632  16640  18326   0.834593  0.524109  0.259169   
1  Dygmamba_global  10382  3632  16640  18326   0.834593  0.524109  0.259169   

       AUC   f_score  
0  0.63247  0.643876  
1  0.63247  0.643876  
active_num: 34966/48980; global num: 34966/48980
benchmark tf gene: 48980, dyg global tf gene: 21958,    merge: 48980
**************************************************
Benchmark TF-Gene: 98,     500, edge: 48980
Dygmamba Merge TF-Gene: 98,     500, edge: 48980
Dygmamba Global Merge TF-Gene: 98,     500, edge: 48980
Dygmamba Global TF-Gene: 98,     246, edge: 21958
Dygmamba active TF-Gene: 98,     246, edge: 21958
**************************************************


# Total Code

In [ ]:
graph_df = pd.read_pickle(dyg_result_path + "Graph_df.pkl")
graph_df["Unnamed"] = graph_df.index
name_list = ["Unnamed", "source_node", "target_node", "time", "label", "edge_idx"]
New_Graph = graph_df[name_list].copy()
New_Graph.columns = ['Unnamed: 0', 'u', 'i', 'ts', 'label', 'idx']


result_path = dyg_result_path + 'my_result_run{run}.npy'
predict_edge_label = np.load(result_path)
New_Graph["predict"] = predict_edge_label
predict_grn = New_Graph.copy()

mapping_series = Node_id["name"]
predict_grn['source'] = (predict_grn['u'] - 1).map(mapping_series)
predict_grn['target'] = (predict_grn['i'] - 1).map(mapping_series)
peak_gene_df = predict_grn[['source', 'target', 'ts','predict']].rename(
    columns={'source': 'Peak', 'target': 'Gene'}
)
peak_gene_df = peak_gene_df[~peak_gene_df["Gene"].str.startswith('chr')].copy()
print("*"*20)
print(peak_gene_df)


jaspar_tf_region_file = org_data_path + "jaspar_data.h5ad"
jaspar_data = ad.read_h5ad(jaspar_tf_region_file)
adata_region_tf = filter_jaspar_tf(jaspar_data)

coo_matrix = adata_region_tf.X.tocoo()
tf_peak_df = pd.DataFrame({
    'Peak': adata_region_tf.obs_names[coo_matrix.row],
    'TF': adata_region_tf.var_names[coo_matrix.col],
    'value': coo_matrix.data
})


merged_df = pd.merge(tf_peak_df, peak_gene_df, on='Peak')
print("*"*50)
print(merged_df)

tf_gene_grn = merged_df.groupby(['TF', 'Gene', 'ts']).agg(
    peak_num=('Peak', 'nunique'),   # 对 Peak 列做去重计数，新列名叫 peak_num
    avg_weight=('predict', 'mean'),  # 对 weight 列做均值，新列名叫 avg_weight
    total_weight=('predict', 'sum')  # (可选) 建议顺便算个总权重
).reset_index()

# 查看结果
print("*"*50)
print(tf_gene_grn)

tf_gene_grn.to_pickle(dyg_result_path + "new_tf_gene_grn.pkl")


avg_active_tf_gene_grn = tf_gene_grn.groupby(['TF', 'Gene']).agg(
    avg_ts_weight=('total_weight', 'mean'),  # 对 weight 列做均值，新列名叫 avg_weight
    avg_total_weight=('total_weight', 'sum')  # (可选) 建议顺便算个总权重
).reset_index()
avg_active_tf_gene_grn.to_pickle(dyg_result_path + "average_active_tf_gene_grn.pkl")

print("*"*50)
print(avg_active_tf_gene_grn)


pivoted_grn = tf_gene_grn.pivot_table(
    index=['TF', 'Gene'],
    columns='ts',
    values='total_weight',
    fill_value=0
)
pivoted_grn['average_active_weight'] = pivoted_grn.mean(axis=1)
avg_global_tf_gene_grn = pivoted_grn.reset_index()
avg_global_tf_gene_grn = avg_global_tf_gene_grn[["TF","Gene","average_active_weight"]].copy()
avg_global_tf_gene_grn.columns.name = None
avg_global_tf_gene_grn.to_pickle(dyg_result_path + "average_global_tf_gene_grn.pkl")

print("*"*50)
print(avg_global_tf_gene_grn)

print("*"*50)
print(f"TF-Peak: {tf_peak_df['TF'].nunique()}, {tf_peak_df['Peak'].nunique()}, edge: {len(tf_peak_df)}")
print(f"Peak-Gene: {peak_gene_df['Peak'].nunique()}, {peak_gene_df['Gene'].nunique()}, edge:{len(peak_gene_df)}")
print(f"TF-Gene: {tf_gene_grn['TF'].nunique()}, {tf_gene_grn['Gene'].nunique()}, edge: {len(tf_gene_grn)}")
print(f"TF-Gene: {avg_active_tf_gene_grn['TF'].nunique()}, {avg_active_tf_gene_grn['Gene'].nunique()}, edge: {len(avg_active_tf_gene_grn)}")

# Further

In [ ]:
output_path = "/home/liyang/BioWuYan/dygmamba_project/data/original/"
adata_rp_gene_peak = ad.read_h5ad(output_path + "binary_peak_gene_rp_network.h5ad")

In [ ]:
print(adata_rp_gene_peak)
from data_preprocess import adata_to_dataframe

prior_peak_gene_df = adata_to_dataframe(adata_rp_gene_peak)

prior_peak_gene_df.head()
prior_peak_gene_df = prior_peak_gene_df.rename(columns= {"obs":"Gene", "var":"Peak"})
print(prior_peak_gene_df)

In [ ]:
output_path = "/home/liyang/BioWuYan/dygmamba_project/data/dygmamba/res/result2/"

adata_rp_gene_peak = ad.read_h5ad(output_path + "rp_gene_peak.h5ad")
print(adata_rp_gene_peak)
prior_peak_gene_df = adata_to_dataframe(adata_rp_gene_peak)

prior_peak_gene_df = prior_peak_gene_df.rename(columns= {"obs":"Gene", "var":"Peak"})
print(prior_peak_gene_df)
